# The Mimic — Phase 3 Healing Fine-Tune (Colab Pro)

Runs Stage 1 (sparse CPT) and Stage 2 (sparse SFT) on Qwen 2.5 0.5B with the sink+window mask active during training.

**Before running**: select an A100 runtime (Runtime → Change runtime type → A100 GPU). T4 will work for the smoke cell but Stage 1/2 are A100-tier.

## 1. Mount Drive & clone repo

Either git-clone (preferred — easy iteration) or mount the local copy from Google Drive.

In [ ]:
import os
USE_GIT = True  # set False to use Google Drive mount instead
REPO_URL = 'https://github.com/<your-user>/the-mimic.git'  # edit this
REPO_PATH = '/content/the-mimic'

if USE_GIT:
    !git clone {REPO_URL} {REPO_PATH}
else:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/the-mimic'  # adjust to your Drive path

os.chdir(REPO_PATH)
!ls

## 2. Install dependencies

Colab's preinstalled `transformers` and `torch` may differ from the repo's pins. We override with the project's `requirements.txt` plus training extras.

In [ ]:
!pip install -q -U transformers peft trl accelerate datasets bitsandbytes wandb pyyaml
import torch, transformers, trl, peft
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, 'trl', trl.__version__, 'peft', peft.__version__)

## 3. Optional — Weights & Biases

Skip if you don't have an account; set `report_to: 'none'` in the YAML configs.

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

## 4. Download Phase 3 data + generate metacognitive seed

Streams the HF datasets per `data/download_phase3.py`. ~10–20 min depending on bandwidth.

In [ ]:
!python data/raw/_metacognitive_seed_v0.py
!python data/download_phase3.py
!ls -lh data/processed/

## 5. Smoke check — verify sparse-mask training step

Runs 50 steps on the CPT data with the sparse collator. Confirms QLoRA + sparse-mask path works before launching the full run.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/heal_cpt.yaml'))
cfg['training']['num_train_epochs'] = 1
cfg['training']['max_steps'] = 50
cfg['training']['report_to'] = 'none'
cfg['training']['output_dir'] = 'outputs/smoke_sparse'
yaml.safe_dump(cfg, open('configs/_smoke.yaml', 'w'))
!python train_sparse.py --config configs/_smoke.yaml

## 6. Stage 1 — Sparse continued pretraining

~6–10 hours on A100. Adapter saves to `outputs/heal_cpt/`. Sync this back to Drive when done.

In [ ]:
!python train_sparse.py --config configs/heal_cpt.yaml

## 7. Stage 2 — Sparse supervised fine-tuning

~4–6 hours on A100. Resumes from the Stage 1 adapter.

In [ ]:
!python train_sparse.py --config configs/heal_sft.yaml

## 8. Eval — sweep on the trained model

Replicates the Phase 1 sweep on the post-FT adapter. Compare PPL ratios against the Phase 1 baseline to quantify the healing effect.

In [ ]:
# TODO: extend eval/sweep.py to accept --adapter-path and merge LoRA before measuring.
# Until then, run the same sweep on the base model (Phase 1) and store the post-FT run separately.
!python -m eval.sweep --n-samples 30 --seq-len 512 --windows 32 64 128 256 --sinks 0 4 8

## 9. Sync outputs back to Drive

Adapters and eval JSON results survive only as long as the Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/drive/MyDrive/the-mimic-outputs/
!cp -r outputs/heal_cpt /content/drive/MyDrive/the-mimic-outputs/
!cp -r outputs/heal_sft /content/drive/MyDrive/the-mimic-outputs/
!cp -r eval/results /content/drive/MyDrive/the-mimic-outputs/